# Cell 1 — Imports and setup

In [1]:
import os
import re
import sqlite3
import pandas as pd
import chromadb

from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

from chromadb.utils import embedding_functions
from semantic_router import Route
from semantic_router.routers import SemanticRouter
from semantic_router.encoders import HuggingFaceEncoder

D:\Gen AI\Project 3\ecommerce-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(Path("..") / ".env")

GROQ_MODEL = os.getenv("GROQ_MODEL")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)

db_path = Path("../data/ecommerce.db")

## Cell 2 — Load ChromaDB

In [3]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

chroma_client = chromadb.PersistentClient(
    path="../data/chroma_db"
)

faq_collection = chroma_client.get_collection(
    name="faqs",
    embedding_function=ef
)

print("FAQ count:", faq_collection.count())

2026-09-30 23:42:14 - huggingface_hub.utils._http - WARNING - _http.py:993 - _warn_on_warning_headers() - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-30 23:42:14 - sentence_transformers.base.model - INFO - model.py:1080 - _load_modules() - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1713.11it/s]


FAQ count: 10


## Cell 3 — FAQ chain

In [4]:
def get_relevant_faqs(query, top_k=3):
    return faq_collection.query(
        query_texts=[query],
        n_results=top_k
    )

In [5]:
def build_faq_context(results):
    parts = []

    for question, metadata in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        parts.append(
            f"Question: {question}\n"
            f"Answer: {metadata['answer']}"
        )

    return "\n\n".join(parts)

In [6]:
def faq_chain(query):
    results = get_relevant_faqs(query)

    context = build_faq_context(results)

    prompt = f"""
You are an e-commerce customer support assistant.

Answer the question using ONLY the FAQ context below.

If the answer is not available in the context, say:
"I don't know based on the available FAQ information."

FAQ CONTEXT:
{context}

USER QUESTION:
{query}
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

## Cell 4 — SQL chain

In [7]:
sql_prompt = """
You convert natural-language product questions into SQLite queries.

Table: product

Columns:
- product_link
- title
- brand
- price
- discount
- avg_rating
- total_ratings

Rules:
1. Generate only SELECT queries.
2. Use SELECT * unless aggregation is required.
3. Use the product table only.
4. For brand matching use:
   LOWER(brand) LIKE LOWER('%brand%')
5. Use ORDER BY where relevant.
6. Limit normal product-list queries to 10 rows unless the user asks for another number.
7. Never generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, TRUNCATE, or REPLACE.
8. Return SQL only inside <SQL></SQL>.
"""

In [8]:
def generate_sql_query(question):
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": sql_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [9]:
def extract_sql(response):
    match = re.search(
        r"<SQL>(.*?)</SQL>",
        response,
        re.DOTALL
    )

    return match.group(1).strip() if match else None

In [10]:
def is_safe_sql(query):
    if not query:
        return False

    cleaned = query.strip().upper()

    forbidden = [
        "INSERT", "UPDATE", "DELETE",
        "DROP", "ALTER", "CREATE",
        "TRUNCATE", "REPLACE"
    ]

    return (
        cleaned.startswith("SELECT")
        and not any(word in cleaned for word in forbidden)
    )

In [11]:
def run_query(query):
    if not is_safe_sql(query):
        raise ValueError("Unsafe SQL blocked.")

    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)

In [12]:
def generate_product_answer(question, products):
    prompt = f"""
You are an e-commerce assistant.

Use ONLY the product data provided.

For each relevant product include:
- title
- brand
- price in INR
- discount percentage
- rating
- product link

QUESTION:
{question}

DATA:
{products}
"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [13]:
def sql_chain(question):
    raw_sql = generate_sql_query(question)

    sql_query = extract_sql(raw_sql)

    if not sql_query:
        return "I couldn't generate a valid product query."

    if not is_safe_sql(sql_query):
        return "The generated query was blocked for safety."

    result_df = run_query(sql_query)

    if result_df.empty:
        return "No matching products were found."

    products = result_df.to_dict(orient="records")

    return generate_product_answer(
        question,
        products
    )

## Cell 5 — Small-talk chain

In [14]:
def smalltalk_chain(query):
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a friendly e-commerce chatbot assistant. "
                    "Keep small-talk responses short and helpful."
                )
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

## Cell 6 — Build the semantic router

In [15]:
encoder = HuggingFaceEncoder(
    name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2102.00it/s]


In [16]:
faq_route = Route(
    name="faq",
    utterances=[
        "What is your return policy?",
        "How can I track my order?",
        "What payment methods are accepted?",
        "How long does a refund take?",
        "Can I cancel my order?",
        "Do you offer international shipping?",
        "What should I do if my product is damaged?"
    ]
)

product_route = Route(
    name="product",
    utterances=[
        "Show me Nike shoes under 5000",
        "What are the cheapest shoes available?",
        "Show products with rating above 4.5",
        "Which Campus shoes have the highest discount?",
        "Suggest Puma shoes under 3000",
        "Show me the top rated products"
    ]
)

smalltalk_route = Route(
    name="smalltalk",
    utterances=[
        "Hello",
        "How are you?",
        "What is your name?",
        "Who are you?",
        "Are you a robot?",
        "What can you do?"
    ]
)

In [17]:
router = SemanticRouter(
    encoder=encoder,
    routes=[
        faq_route,
        product_route,
        smalltalk_route
    ],
    auto_sync="local"
)

router.set_threshold(0.40)

2026-10-01 00:05:42 WARNING semantic_router No index provided. Using default LocalIndex.
2026-10-01 00:05:42 WARNING semantic_router No config is written for LocalIndex.


## Cell 7 — Build the orchestrator

In [34]:
def ask(query):
    route = router(query).name

    print("Route:", route)

    if route == "faq":
        return faq_chain(query)

    elif route == "product":
        return sql_chain(query)

    elif route == "smalltalk":
        return smalltalk_chain(query)

    else:
        return (
                "I'm not sure how to handle that request. "
                "I can help with products, orders, returns, payments, "
                "shipping, or e-commerce related questions."
    
                )

router decides
     ↓
ask() orchestrates
     ↓
specialized chain executes

## Cell 8 — Test the complete system

In [35]:
test_queries = [
    "My shoes arrived damaged. What should I do?",
    "Show me 5 shoes under 1500 rupees",
    "Hey, how are you?",
    "Explain quantum physics"
]

for query in test_queries:
    print("\n" + "=" * 80)
    print("USER:", query)
    print("\nBOT:")
    print(ask(query))


USER: My shoes arrived damaged. What should I do?

BOT:
Route: faq


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.41it/s]


Contact our support team within 48 hours for a replacement or refund.

USER: Show me 5 shoes under 1500 rupees

BOT:
Route: product
**5 Shoes (≤ ₹1500)**  

| Title | Brand | Price (₹) | Discount | Rating | Product Link |
|-------|-------|-----------|----------|--------|--------------|
| SUPEX women & girls casual wear comfort shoes Walking Shoes For Women | Supex | 191 | 68 % | 4.1 | https://www.flipkart.com/supex-women-girls-casual-wear-comfort-shoes-walking/p/itmc5e3bfb23bb29?pid=SHOGJ5FHDTNGFG8U&lid=LSTSHOGJ5FHDTNGFG8UIFNNDR&marketplace=FLIPKART&q=sports+shoes+for+women&store=osp%2Fiko%2Fd20&srno=s_25_969&otracker=search&otracker1=search&fm=organic&iid=bdb520b3-a976-4fc4-b0ca-36539eef9b74.SHOGJ5FHDTNGFG8U.SEARCH&ppt=sp&ppn=sp&ssid=tawn8xhdww0000001716205040752&qH=32e27b1148959538 |
| Mesh |Lightweight|Comfort|Trendy|Walking|Outdoor|Daily Use Running Shoes For Women | FEEL TIME | 249 | 75 % | 4.4 | https://www.flipkart.com/feel-time-mesh-lightweight-comfort-trendy-walking-outdoor-da